# Day 084 — Exercise 1: The Researcher Agent

**What you'll build:** `build_researcher_prompt` and `ResearcherAgent` — a specialist whose only job is to gather facts on a topic and return them as a string.

**Why it matters:** specialisation is the core idea of multi-agent systems. A researcher that only researches, and a writer that only writes, each does its job better than one agent trying to do both. The specialist pattern is also reusable: a research agent can feed a planner, a writer, or a reviewer — because its interface is a simple `research(query) -> str`.

In [ ]:
def _mock_multi(findings='Finding: AI agents collaborate.', document='Doc: Agents work together.'):
    """Branch on system message: 'research specialist' -> findings, else -> document."""
    def _fn(messages):
        system = messages[0]['content'] if messages else ''
        if 'research specialist' in system.lower():
            return findings
        return document
    return _fn
import json

# ── helpers reused from Day 79 ───────────────────────────────────────────────
def safe_parse_json(text):
    """Slice first '{' to last '}' and parse. Returns dict|None (Day 79)."""
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end < start:
        return None
    try:
        data = json.loads(text[start:end + 1])
    except (json.JSONDecodeError, ValueError):
        return None
    return data if isinstance(data, dict) else None


def call_llm(messages, llm_fn=None):
    """Call the chat model, or the injected llm_fn(messages) -> str (Day 79)."""
    if llm_fn is not None:
        return llm_fn(messages)
    import ollama
    resp = ollama.chat(model="llama3.2", messages=messages)
    return resp["message"]["content"]


## Task

1. `build_researcher_prompt(query)` — a `system` message orienting the agent as a research specialist (gather facts, be concise and factual); a `user` message: `'Research topic: ' + query`.
2. `ResearcherAgent(llm_fn=None)` — `research(query)` calls `call_llm` with the prompt and records `{'query', 'findings'}` in `_history`; returns the findings string. `history()` returns a copy; `clear_history()` empties in place.

## Your Implementation

In [ ]:
def build_researcher_prompt(query):
    """Build a prompt for the researcher role: gather facts on a topic."""
    raise NotImplementedError

class ResearcherAgent:
    """A specialist that researches a topic and returns structured findings."""

    def __init__(self, llm_fn=None):
        raise NotImplementedError

    def research(self, query):
        raise NotImplementedError

    def history(self):
        raise NotImplementedError

    def clear_history(self):
        raise NotImplementedError


In [ ]:

# ── researcher specialist ─────────────────────────────────────────────────────
def build_researcher_prompt(query):
    """Build a prompt for the researcher role: gather facts on a topic."""
    system = "\n".join([
        "You are a research specialist. Your job is to gather relevant facts and",
        "key information about the topic given to you.",
        "",
        "Return a structured list of the most important findings.",
        "Be factual, concise, and cover the main points.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": "Research topic: " + str(query)}]


class ResearcherAgent:
    """A specialist that researches a topic and returns structured findings.

    Each call to research() returns a string of findings and records the
    exchange in history. The agent has one job: gather facts. It passes its
    output to the next agent via a Handoff — it does not write, review, or
    plan.

    Example::

        researcher = ResearcherAgent(llm_fn=my_llm_fn)
        findings = researcher.research("topological sort algorithms")
    """

    def __init__(self, llm_fn=None):
        self._llm_fn = llm_fn
        self._history = []

    def research(self, query):
        """Research a query and return findings as a string."""
        messages = build_researcher_prompt(query)
        findings = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"query": query, "findings": findings})
        return findings

    def history(self):
        """Return a copy of the research history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()


## Automated checks

In [ ]:

score, total = 0, 5
try:
    msgs = build_researcher_prompt('topological sort')
    assert msgs[0]['role'] == 'system' and 'research' in msgs[0]['content'].lower()
    assert msgs[1]['content'].startswith('Research topic:')
    score += 1; print("✅ build_researcher_prompt has the right shape")

    r = ResearcherAgent(llm_fn=_mock_multi(findings='Fact: X.'))
    out = r.research('AI agents')
    assert out == 'Fact: X.'
    score += 1; print("✅ research() returns the LLM output")

    assert len(r.history()) == 1 and r.history()[0]['query'] == 'AI agents'
    score += 1; print("✅ history records each call")

    r.history().clear()
    assert len(r.history()) == 1
    score += 1; print("✅ history() returns a copy, not the live list")

    r.clear_history()
    assert len(r.history()) == 0
    score += 1; print("✅ clear_history() empties in place")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python

# ── researcher specialist ─────────────────────────────────────────────────────
def build_researcher_prompt(query):
    """Build a prompt for the researcher role: gather facts on a topic."""
    system = "\n".join([
        "You are a research specialist. Your job is to gather relevant facts and",
        "key information about the topic given to you.",
        "",
        "Return a structured list of the most important findings.",
        "Be factual, concise, and cover the main points.",
    ])
    return [{"role": "system", "content": system},
            {"role": "user", "content": "Research topic: " + str(query)}]


class ResearcherAgent:
    """A specialist that researches a topic and returns structured findings.

    Each call to research() returns a string of findings and records the
    exchange in history. The agent has one job: gather facts. It passes its
    output to the next agent via a Handoff — it does not write, review, or
    plan.

    Example::

        researcher = ResearcherAgent(llm_fn=my_llm_fn)
        findings = researcher.research("topological sort algorithms")
    """

    def __init__(self, llm_fn=None):
        self._llm_fn = llm_fn
        self._history = []

    def research(self, query):
        """Research a query and return findings as a string."""
        messages = build_researcher_prompt(query)
        findings = call_llm(messages, llm_fn=self._llm_fn)
        self._history.append({"query": query, "findings": findings})
        return findings

    def history(self):
        """Return a copy of the research history."""
        return list(self._history)

    def clear_history(self):
        """Clear the history in place."""
        self._history.clear()
```

**Why does the researcher *only* research?** Single responsibility makes each agent predictable and replaceable. If the researcher also tried to write, you could not swap it for a different researcher without rewriting both jobs. The clean interface — `research(query) -> str` — is what lets any downstream agent (writer, planner, reviewer) consume its output.

</details>